# Sipat Kilatis: fine-tune RoBERTa-tagalog on Colab

Backup for laptops without an NVIDIA GPU. Runs the same scripts as the laptop (`train_transformer.py`, `export_onnx.py`).

1. **Runtime > Change runtime type > T4 GPU**
2. Run the cells in order. Cell 3 asks you to upload `ml/data/processed/messages.csv` from your laptop
   (the dataset is not on GitHub on purpose: it contains real messages).
3. The last cell downloads `sipat_models.zip`. Unzip it into `ml/models/` on your laptop.

In [ ]:
# 1. Get the code
!git clone https://github.com/MCQuisao/Sipat-Kilatis.git
%cd Sipat-Kilatis/ml

In [ ]:
# 2. Install packages (Colab already has a CUDA build of torch)
!pip install -q "transformers>=4.45,<4.58" datasets accelerate evaluate "optimum[onnxruntime]" onnx onnxruntime skl2onnx
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU')

In [ ]:
# 3. Upload messages.csv (from ml/data/processed/ on your laptop)
import os, shutil
from google.colab import files
uploaded = files.upload()
os.makedirs('data/processed', exist_ok=True)
shutil.move(next(iter(uploaded)), 'data/processed/messages.csv')
print('OK:', os.path.getsize('data/processed/messages.csv'), 'bytes')

In [ ]:
# 4. Fine-tune (about 2-5 minutes on a T4)
!python train_transformer.py

In [ ]:
# 5. Export to ONNX, quantize to int8, check, write test vectors
!python export_onnx.py

In [ ]:
# 6. Download the files the app needs
!cd models && zip -qr ../sipat_models.zip scam_classifier_int8.onnx tokenizer test_vectors.json transformer_metrics.json transformer_export.json
files.download('sipat_models.zip')